# XGBoost Classification for TLS Profiling

This notebook evaluates a supervised XGBoost baseline on extracted TLS traffic features. The dataset paths are parameterized so the same experiment can be run on one or more parquet partitions. Unlike the anomaly-detection baselines, this model is trained to classify each flow directly into one of four categories: `system`, `malware`, `application`, or `unknown`.


In [ ]:
import sys
from pathlib import Path
sys.path.append('../../src')

### PARAMETERS:


In [8]:
train_features_path = ["../data-ext/malware_train.parquet", "../data-ext/apps_train.parquet"]
val_features_path = ["../data-ext/malware_val.parquet", "../data-ext/apps_val.parquet"]
test_features_path = ["../data-ext/malware_test.parquet", "../data-ext/apps_test.parquet"]
train_labels_path = ["../data-ext/malware_train_labels.parquet", "../data-ext/apps_train_labels.parquet"]
val_labels_path = ["../data-ext/malware_val_labels.parquet", "../data-ext/apps_val_labels.parquet"]
test_labels_path = ["../data-ext/malware_test_labels.parquet", "../data-ext/apps_test_labels.parquet"]


## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The experiments below sweep over every non-empty combination of these groups.


In [9]:
feature_groups = {
    "FLOW": ["bs", "ps", "br", "pr", "td"],
    "CTLS": ["tls.cver", "tls.ccs", "tls.cext", "tls.csg", "tls.alpn", "tls.csv"],
    "STLS": ["tls.sver", "tls.scs", "tls.sext", "tls.ssv"],
    "REC": ["tls.rec"],
}

FLOW = feature_groups["FLOW"]
CTLS = feature_groups["CTLS"]
STLS = feature_groups["STLS"]
REC = feature_groups["REC"]


In [4]:
import pandas as pd
from pathlib import Path
from tls_profiling.preprocessing import build_and_fit_preprocessor

def read_parquet_files(paths):
    if isinstance(paths, (str, Path)):
        paths = [paths]
    return pd.concat([pd.read_parquet(Path(path)) for path in paths], ignore_index=True)

print("Loading extracted features from parameterized parquet paths.")
df_train = read_parquet_files(train_features_path)
df_val = read_parquet_files(val_features_path)
df_test = read_parquet_files(test_features_path)
df_train_label = read_parquet_files(train_labels_path)
df_val_label = read_parquet_files(val_labels_path)
df_test_label = read_parquet_files(test_labels_path)

preprocessor = build_and_fit_preprocessor(df_train)
print("Built preprocessor from df_train.")


Loading extracted features from parameterized parquet paths.
Built preprocessor from df_train.


In [10]:
from tls_profiling.baselines.model_xgboost import XGBoostBaseline, Config
import numpy as np

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

CLASS_NAMES = ["system", "malware", "application", "unknown"]
CLASS_TO_INDEX = {label: idx for idx, label in enumerate(CLASS_NAMES)}
INDEX_TO_CLASS = {idx: label for label, idx in CLASS_TO_INDEX.items()}

XGB_CONFIG_CANDIDATES = {
    "balanced_hist_deep": Config(
        n_estimators=400,
        max_depth=8,
        learning_rate=0.05,
        min_child_weight=1.0,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        max_train_samples=100_000,
    ),
    "balanced_hist_shallow": Config(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.08,
        min_child_weight=2.0,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        max_train_samples=100_000,
    ),
}

def encode_labels(series: pd.Series) -> np.ndarray:
    return series.map(CLASS_TO_INDEX).to_numpy(dtype=np.int64)

def decode_labels(values: np.ndarray) -> list[str]:
    return [INDEX_TO_CLASS[int(value)] for value in values]

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def compute_multiclass_scores(y_true, y_pred, y_score):
    y_true_bin = label_binarize(y_true, classes=np.arange(len(CLASS_NAMES)))
    per_class_f1 = f1_score(
        y_true,
        y_pred,
        average=None,
        labels=np.arange(len(CLASS_NAMES)),
    )

    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_auc": roc_auc_score(
            y_true,
            y_score,
            labels=np.arange(len(CLASS_NAMES)),
            multi_class="ovr",
            average="macro",
        ),
        "macro_ap": average_precision_score(y_true_bin, y_score, average="macro"),
        "system_f1": per_class_f1[CLASS_TO_INDEX["system"]],
        "malware_f1": per_class_f1[CLASS_TO_INDEX["malware"]],
        "application_f1": per_class_f1[CLASS_TO_INDEX["application"]],
        "unknown_f1": per_class_f1[CLASS_TO_INDEX["unknown"]],
    }

def choose_best_model(x_train, y_train, x_val, y_val):
    best_name = None
    best_model = None
    best_val_macro_f1 = -np.inf

    for config_name, config in XGB_CONFIG_CANDIDATES.items():
        model = XGBoostBaseline(config)
        model.fit(x_train, y_train)

        val_pred = model.predict(x_val)
        val_macro_f1 = f1_score(y_val, val_pred, average="macro")

        if val_macro_f1 > best_val_macro_f1:
            best_name = config_name
            best_model = model
            best_val_macro_f1 = val_macro_f1

    return best_name, best_model, float(best_val_macro_f1)

def evaluate(feature_set):
    y_train = encode_labels(df_train_label["category"])
    y_val = encode_labels(df_val_label["category"])
    y_test = encode_labels(df_test_label["category"])

    x_train_transformed = select_feature_set(preprocessor.transform(df_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(df_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(df_test), feature_set)

    best_config_name, model, val_macro_f1 = choose_best_model(
        x_train_transformed,
        y_train,
        x_val_transformed,
        y_val,
    )

    y_pred = model.predict(x_test_transformed)
    y_score = model.predict_proba(x_test_transformed)

    return y_test, y_pred, y_score, val_macro_f1, best_config_name

def debug_csv(feature_set, y_test, y_pred, y_score):
    feature_set_name = "_".join(feature_set)
    output_path = f"tmp/xgb_{feature_set_name}.csv"

    pd.DataFrame({
        "y_test": decode_labels(y_test),
        "y_pred": decode_labels(y_pred),
        "p_system": y_score[:, CLASS_TO_INDEX["system"]],
        "p_malware": y_score[:, CLASS_TO_INDEX["malware"]],
        "p_application": y_score[:, CLASS_TO_INDEX["application"]],
        "p_unknown": y_score[:, CLASS_TO_INDEX["unknown"]],
    }).to_csv(output_path, index=False)

def compute_scores(feature_set):
    y_test, y_pred, y_score, val_macro_f1, best_config_name = evaluate(feature_set)
    scores = compute_multiclass_scores(y_test, y_pred, y_score)

    debug_csv(feature_set, y_test, y_pred, y_score)
    return {
        "best_config": best_config_name,
        "val_macro_f1": val_macro_f1,
        **scores,
    }


## Evaluation

This baseline uses the same train, validation, and test splits as the anomaly-detection notebooks, but the task is now multiclass classification rather than one-class profiling.

Protocol:

- `train`: fit XGBoost on labeled traffic from all target categories
- `val`: select the best hyperparameter configuration using macro F1
- `test`: report multiclass metrics for the selected configuration on held-out traffic

The target classes in this notebook are `system`, `malware`, `application`, and `unknown`.


In [11]:
from itertools import combinations

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ", ".join(group_combo)
        scores = compute_scores(selected_features)

        rows.append({
            "FeatureSet": feature_set_name,
            "BestConfig": scores["best_config"],
            "ValMacroF1": scores["val_macro_f1"],
            "MacroF1": scores["macro_f1"],
            "MacroAucScore": scores["macro_auc"],
            "MacroApScore": scores["macro_ap"],
            "SystemF1": scores["system_f1"],
            "MalwareF1": scores["malware_f1"],
            "ApplicationF1": scores["application_f1"],
            "UnknownF1": scores["unknown_f1"],
        })

results_df = pd.DataFrame(
    rows,
    columns=[
        "FeatureSet",
        "BestConfig",
        "ValMacroF1",
        "MacroF1",
        "MacroAucScore",
        "MacroApScore",
        "SystemF1",
        "MalwareF1",
        "ApplicationF1",
        "UnknownF1",
    ],
)
display(results_df)


/home/rysavy/devel/GitHub/AutoFedProfile/.venv/lib/python3.12/site-packages/pandas/core/base.py:666: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)
/home/rysavy/devel/GitHub/AutoFedProfile/.venv/lib/python3.12/site-packages/pandas/core/base.py:666: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)
/home/rysavy/devel/GitHub/AutoFedProfile/.venv/lib/python3.12/site-packages/pandas/core/base.py:666: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2 3], got [-9223372036854775808                    0                    1
                    2]

## Overall Results

The table below ranks feature sets by `MacroF1`, which is the primary metric for this multiclass baseline. `MacroAucScore` and `MacroApScore` are included to make the supervised baseline easier to compare at a high level with the anomaly-detection notebooks.


In [ ]:
overall_df = results_df.sort_values("MacroF1", ascending=False)
display(overall_df)

## System Category

This table isolates how well each feature set classifies the `system` category. A high `SystemF1` means the supervised classifier can separate system traffic cleanly from both malware and application traffic.


In [ ]:
system_df = results_df.sort_values("SystemF1", ascending=False)
display(system_df)

## Malware Category

This section focuses on the `malware` class. Strong `MalwareF1` values indicate that the selected feature family helps XGBoost carve out malware traffic as a distinct supervised class rather than only separating it from one chosen normal profile.


In [ ]:
malware_df = results_df.sort_values("MalwareF1", ascending=False)
display(malware_df)


## Application Category

This section focuses on the `application` class. Strong `ApplicationF1` values indicate that the selected feature family helps XGBoost separate application traffic from both system and malware flows in the combined dataset.


In [ ]:
application_df = results_df.sort_values("ApplicationF1", ascending=False)
display(application_df)


## SHAP Interpretation

This section retrains the best overall XGBoost configuration and visualizes SHAP values for the `malware` class on a sample of the held-out test split. The goal is to show which transformed features most strongly push predictions toward the malware class in the best-performing supervised model.


In [ ]:
import matplotlib.pyplot as plt
import shap

def expand_feature_groups(group_combo):
    selected_features = []
    for group_name in group_combo:
        selected_features.extend(feature_groups[group_name])
    return selected_features

def extract_class_shap_values(shap_values, class_index):
    values = getattr(shap_values, "values", shap_values)
    if isinstance(values, list):
        return np.asarray(values[class_index])

    values = np.asarray(values)
    if values.ndim == 3:
        return values[:, :, class_index]
    if values.ndim == 2:
        return values

    raise ValueError(f"Unsupported SHAP value shape: {values.shape}")

best_row = overall_df.iloc[0]
best_group_combo = tuple(best_row["FeatureSet"].split(", "))
best_feature_set = expand_feature_groups(best_group_combo)

x_train_best = select_feature_set(preprocessor.transform(df_train), best_feature_set)
x_test_best = select_feature_set(preprocessor.transform(df_test), best_feature_set)
y_train_best = encode_labels(df_train_label["category"])

best_model = XGBoostBaseline(XGB_CONFIG_CANDIDATES[best_row["BestConfig"]])
best_model.fit(x_train_best, y_train_best)

background = x_train_best.sample(n=min(1000, len(x_train_best)), random_state=42)
explain_data = x_test_best.sample(n=min(2000, len(x_test_best)), random_state=42)

explainer = best_model.build_shap_explainer(background)
shap_values = best_model.explain(explain_data, explainer=explainer)

class_name = "malware"
class_index = CLASS_TO_INDEX[class_name]
class_shap_values = extract_class_shap_values(shap_values, class_index)

print(
    f"SHAP summary for feature set={best_row['FeatureSet']} "
    f"with config={best_row['BestConfig']} on class={class_name}."
)

plt.figure(figsize=(12, 8))
shap.summary_plot(
    class_shap_values,
    explain_data,
    feature_names=explain_data.columns.tolist(),
    show=False,
)
plt.title(f"SHAP Summary for '{class_name}' Class")
plt.tight_layout()
plt.show()
